In [1]:
!pip install ultralytics roboflow optuna deep-sort-realtime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 667.0 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 3.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 24.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 46.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 52.8 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88


In [2]:
from pathlib import Path
import re

import cv2
from ultralytics import RTDETR
from deep_sort_realtime.deepsort_tracker import DeepSort

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
!pip -q install gdown
!gdown --fuzzy "https://drive.google.com/file/d/19ZzfsW61szg-IU5VSTz-ljmOcvXROzCa/view?usp=drive_link"

Downloading...
From (original): https://drive.google.com/uc?id=19ZzfsW61szg-IU5VSTz-ljmOcvXROzCa
From (redirected): https://drive.google.com/uc?id=19ZzfsW61szg-IU5VSTz-ljmOcvXROzCa&confirm=t&uuid=f91a808a-54f8-49b3-980e-afe9cf72ccfe
To: /kaggle/working/best.pt
100%|███████████████████████████████████████| 66.2M/66.2M [00:00<00:00, 206MB/s]


# For Annotated frames

In [ ]:
from pathlib import Path
import re
import cv2

from ultralytics import RTDETR
from deep_sort_realtime.deepsort_tracker import DeepSort


FRAMES_DIR = Path("/kaggle/working/UAVRoundAboutDetectionTracking-1/train/images")
ANNOTATED_DIR = Path("annotated_frames")
MOT_OUT = Path("results.txt")

ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)

# Collect frames
frame_paths = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
    frame_paths.extend(FRAMES_DIR.glob(ext))

frame_re = re.compile(r"frame_(\d+)", re.IGNORECASE)

def frame_number(p: Path) -> int:
    m = frame_re.search(p.stem)
    if not m:
        raise ValueError(f"Filename does not match 'frame_00001' pattern: {p.name}")
    return int(m.group(1))

frame_paths = sorted(frame_paths, key=frame_number)

detector = RTDETR("/kaggle/working/best.pt")
tracker = DeepSort(max_age=30)

mot_lines = []

# track_id -> last known metadata (so label/color persists even if track has no matched det this frame)
track_meta = {}  # {track_id: {"cls_id": int, "cls_name": str, "conf": float}}

MOT_1_BASED = True
CONF_THRES = 0.25

# -------------------------
# COLORS (OpenCV uses BGR)
# -------------------------
PALETTE = [
    (255,  56,  56),
    ( 56, 255,  56),
    ( 56,  56, 255),
    (255, 157,  56),
    (156,  56, 255),
    ( 56, 255, 255),
    (255,  56, 157),
    (157, 255,  56),
]

def color_for_class(cls_id, cls_name=None):
    if cls_id is None or int(cls_id) < 0:
        return (200, 200, 200)
    return PALETTE[int(cls_id) % len(PALETTE)]


for img_path in frame_paths:
    frame_idx = frame_number(img_path)

    frame = cv2.imread(str(img_path))
    if frame is None:
        continue

    results = detector(str(img_path))
    r = results[0]

    dets = []
    others = []  # supplementary info aligned to dets

    for xyxy, conf, cls in zip(r.boxes.xyxy, r.boxes.conf, r.boxes.cls):
        conf = float(conf)
        if conf < CONF_THRES:
            continue

        cls_id = int(cls)

        if isinstance(r.names, dict):
            cls_name = r.names.get(cls_id, str(cls_id))
        else:
            cls_name = r.names[cls_id]

        x1, y1, x2, y2 = map(float, xyxy)
        w = x2 - x1
        h = y2 - y1

        dets.append(([x1, y1, w, h], conf, cls_id))
        others.append({"cls_id": cls_id, "cls_name": cls_name, "conf": conf})

    tracks = tracker.update_tracks(dets, frame=frame, others=others)

    for trk in tracks:
        if not trk.is_confirmed():
            continue

        track_id = trk.track_id
        l, t, rgt, btm = trk.to_ltrb()

        # Update cache from supplementary detection info (if available)
        supp = trk.get_det_supplementary()
        if isinstance(supp, dict) and ("cls_id" in supp or "cls_name" in supp):
            prev = track_meta.get(track_id, {})
            track_meta[track_id] = {
                "cls_id": int(supp.get("cls_id", prev.get("cls_id", -1))),
                "cls_name": str(supp.get("cls_name", prev.get("cls_name", "unknown"))),
                "conf": float(supp.get("conf", prev.get("conf", 1.0))),
            }

        meta = track_meta.get(track_id, {"cls_id": -1, "cls_name": "unknown", "conf": 1.0})
        cls_id = meta["cls_id"]
        cls_name = meta["cls_name"]
        det_conf = meta["conf"]

        color = color_for_class(cls_id, cls_name)

        x1i, y1i, x2i, y2i = map(int, [l, t, rgt, btm])
        cv2.rectangle(frame, (x1i, y1i), (x2i, y2i), color, 2)

        # Display ONLY the class name (no ID)
        # cv2.putText(
        #     frame,
        #     f"{cls_name}",
        #     (x1i, max(0, y1i - 7)),
        #     cv2.FONT_HERSHEY_SIMPLEX,
        #     0.6,
        #     color,
        #     2,
        #     cv2.LINE_AA,
        # )
        
        # Display class name + confidence score
        # cv2.putText(
        #     frame,
        #     f"{cls_name} {det_conf:.2f}",
        #     (x1i, max(0, y1i - 7)),
        #     cv2.FONT_HERSHEY_SIMPLEX,
        #     0.6,
        #     color,
        #     2,
        #     cv2.LINE_AA
        # )

        label_text = f"{cls_name} {det_conf:.2f}"
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.4
        thickness = 1
        (tw, th), baseline = cv2.getTextSize(label_text, font, font_scale, thickness)
        lx1 = x1i
        ly2 = max(th + baseline + 4, y1i)
        ly1 = ly2 - th - baseline - 4
        lx2 = lx1 + tw + 6
        cv2.rectangle(frame, (lx1, ly1), (lx2, ly2), color, 1)
        cv2.putText(frame, label_text, (lx1 + 3, ly2 - 3), font, font_scale, (0, 0, 0), thickness, cv2.LINE_AA)

        # MOTChallenge 10-column format
        bb_left = float(l)
        bb_top = float(t)
        bb_width = float(rgt - l)
        bb_height = float(btm - t)

        if MOT_1_BASED:
            bb_left += 1.0
            bb_top += 1.0

        mot_lines.append(
            f"{frame_idx},{track_id},{bb_left:.2f},{bb_top:.2f},{bb_width:.2f},{bb_height:.2f},{det_conf:.6f},-1,-1,-1"
        )

    out_path = ANNOTATED_DIR / img_path.name
    cv2.imwrite(str(out_path), frame)

MOT_OUT.write_text("\n".join(mot_lines), encoding="utf-8")
print(f"Annotated frames saved to: {ANNOTATED_DIR}")
print(f"MOT results saved to: {MOT_OUT}  (lines: {len(mot_lines)})")


In [ ]:
# !zip -r annotated_frames.zip /kaggle/working/annotated_frames

# For Annotated Tracking Video

In [7]:
from pathlib import Path
import subprocess
import cv2

from ultralytics import RTDETR
from deep_sort_realtime.deepsort_tracker import DeepSort


# =========================
# PATHS / CONFIG
# =========================
VIDEO_IN  = Path("/kaggle/input/datasets/huzaifasohail321/roundabout-near-2-15fps/RoundaboutNear2_15fps.mp4") 
WEIGHTS   = Path("/kaggle/working/best.pt")

# OpenCV temp output + final re-encoded output (best for Kaggle playback)
VIDEO_OUT_TMP   = Path("/kaggle/working/tracked_RoundaboutNear2_15fps.mp4")
VIDEO_OUT_FINAL = Path("/kaggle/working/tracked_Splitted_RoundaboutNear2_15fps_final_h264.mp4")

CONF_THRES = 0.25
IMGSZ = 640

# To avoid “duplicate looking” boxes, default is: show tracks only
DRAW_DETECTIONS = False            # set True only for debugging detector output
DRAW_UNCONFIRMED_TRACKS = False    # set True if you want to see tentative tracks too

# NMS to reduce duplicate detections feeding DeepSORT
NMS_IOU_THRES = 0.50               # increase to keep more boxes, decrease to suppress more

# Drop weird tiny boxes (often look like lines)
MIN_BOX_PX = 4

# MOT optional
WRITE_MOT = False
MOT_OUT = Path("/kaggle/working/results.txt")
MOT_1_BASED = True


# =========================
# COLORS (OpenCV uses BGR)
# Consistent color per class (what you asked for)
# =========================
CLASS_COLORS = {
    "bus":         (255,   0,   0),  # blue
    "car":         (  0, 255,   0),  # green
    "truck":       (  0,   0, 255),  # red (your current model names dict may not include this)
    "two_wheeler": (  0, 255, 255),  # yellow
    "van":         (255,   0, 255),  # magenta
}
DEFAULT_COLOR = (200, 200, 200)


def draw_label(img, text, x, y, color):
    """Readable label with filled background."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.6
    thickness = 2
    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)

    x1 = int(x)
    y2 = int(max(0, y))
    y1 = int(max(0, y2 - th - baseline - 6))
    x2 = int(x1 + tw + 8)

    cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)
    cv2.putText(img, text, (x1 + 4, y2 - 4), font, font_scale, (0, 0, 0), thickness, cv2.LINE_AA)


def nms_per_class(boxes_xywh, scores, cls_ids, score_thres, iou_thres):
    """
    Per-class NMS using cv2.dnn.NMSBoxes.
    boxes_xywh: list of [x, y, w, h] floats
    scores: list of floats
    cls_ids: list of ints
    returns: list of kept indices (into the original lists)
    """
    keep = []

    if not boxes_xywh:
        return keep

    unique_classes = sorted(set(cls_ids))
    for cid in unique_classes:
        idxs = [i for i, c in enumerate(cls_ids) if c == cid]
        b = [boxes_xywh[i] for i in idxs]
        s = [scores[i] for i in idxs]

        # cv2.dnn.NMSBoxes expects boxes in xywh, can be float; some builds prefer ints
        b_int = [[int(x), int(y), int(w), int(h)] for (x, y, w, h) in b]

        picked = cv2.dnn.NMSBoxes(
            b_int,
            s,
            score_threshold=score_thres,
            nms_threshold=iou_thres,
        )

        if picked is None or len(picked) == 0:
            continue

        # picked can be [[0],[2],...] or [0,2,...] depending on OpenCV version
        picked_flat = []
        for p in picked:
            if isinstance(p, (list, tuple)):
                picked_flat.append(int(p[0]))
            else:
                picked_flat.append(int(p))

        keep.extend([idxs[j] for j in picked_flat])

    keep.sort()
    return keep


# =========================
# LOAD MODEL + TRACKER
# =========================
detector = RTDETR(str(WEIGHTS))

# n_init=2 reduces quick duplicate track creation vs n_init=1
tracker = DeepSort(max_age=30, n_init=2)

cap = cv2.VideoCapture(str(VIDEO_IN))
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_IN}")

# Read first frame to get reliable frame size
ok, first_frame = cap.read()
if not ok or first_frame is None:
    cap.release()
    raise RuntimeError("Could not read first frame from the input video.")

H, W = first_frame.shape[:2]

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps <= 0:
    fps = 25.0

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(VIDEO_OUT_TMP), fourcc, fps, (W, H))
if not writer.isOpened():
    cap.release()
    raise RuntimeError(f"Could not open VideoWriter for: {VIDEO_OUT_TMP}")

# track_id -> {"cls_id","cls_name","conf"}
track_meta = {}
mot_lines = []
frame_idx = 0


def process_one_frame(frame):
    """Run detector+tracker; return annotated frame (BGR)."""
    global frame_idx, track_meta, mot_lines

    frame_idx += 1

    # IMPORTANT: keep tracker input clean (no drawings)
    frame_vis = frame.copy()

    # Detector
    results = detector(frame, imgsz=IMGSZ, verbose=False)
    r = results[0]

    boxes_xywh = []
    scores = []
    cls_ids = []
    cls_names = []

    # Collect raw detections
    if r.boxes is not None and len(r.boxes) > 0:
        for xyxy, conf, cls in zip(r.boxes.xyxy, r.boxes.conf, r.boxes.cls):
            conf = float(conf)
            if conf < CONF_THRES:
                continue

            cls_id = int(cls)

            if isinstance(r.names, dict):
                cls_name = r.names.get(cls_id, str(cls_id))
            else:
                cls_name = r.names[cls_id] if cls_id < len(r.names) else str(cls_id)

            x1, y1, x2, y2 = map(float, xyxy)
            w = x2 - x1
            h = y2 - y1
            if w <= 0 or h <= 0:
                continue

            # drop tiny boxes early (often noisy)
            if w < MIN_BOX_PX or h < MIN_BOX_PX:
                continue

            boxes_xywh.append([x1, y1, w, h])
            scores.append(conf)
            cls_ids.append(cls_id)
            cls_names.append(cls_name)

    # Apply per-class NMS to reduce duplicate detections
    keep = nms_per_class(
        boxes_xywh=boxes_xywh,
        scores=scores,
        cls_ids=cls_ids,
        score_thres=CONF_THRES,
        iou_thres=NMS_IOU_THRES,
    )

    dets = []
    others = []
    for i in keep:
        x1, y1, w, h = boxes_xywh[i]
        conf = scores[i]
        cls_id = cls_ids[i]
        cls_name = cls_names[i]

        dets.append(([x1, y1, w, h], conf, cls_id))
        others.append({"cls_id": cls_id, "cls_name": cls_name, "conf": conf})

        # Optional: draw raw detections (thin) on visualization frame
        if DRAW_DETECTIONS:
            color = CLASS_COLORS.get(cls_name, DEFAULT_COLOR)
            cv2.rectangle(frame_vis, (int(x1), int(y1)), (int(x1 + w), int(y1 + h)), color, 1)
            draw_label(frame_vis, cls_name, int(x1), int(y1), color)

    # Tracker update (pass clean frame, not frame_vis)
    try:
        tracks = tracker.update_tracks(dets, frame=frame, others=others)
    except TypeError:
        tracks = tracker.update_tracks(dets, frame=frame)

    confirmed_count = 0

    # Draw tracks (thick)
    for trk in tracks:
        if trk.is_confirmed():
            confirmed_count += 1

        if (not DRAW_UNCONFIRMED_TRACKS) and (not trk.is_confirmed()):
            continue

        # Skip stale tracks not updated this frame
        if hasattr(trk, "time_since_update") and trk.time_since_update > 0:
            continue

        track_id = trk.track_id
        l, t, rgt, btm = trk.to_ltrb()

        # Clamp to frame and validate
        l = max(0.0, min(float(l), W - 1.0))
        t = max(0.0, min(float(t), H - 1.0))
        rgt = max(0.0, min(float(rgt), W - 1.0))
        btm = max(0.0, min(float(btm), H - 1.0))

        bw = rgt - l
        bh = btm - t
        if bw < MIN_BOX_PX or bh < MIN_BOX_PX:
            continue

        # Update cache from supplementary info (if available)
        supp = trk.get_det_supplementary() if hasattr(trk, "get_det_supplementary") else None
        if isinstance(supp, dict) and ("cls_id" in supp or "cls_name" in supp):
            prev = track_meta.get(track_id, {})
            track_meta[track_id] = {
                "cls_id": int(supp.get("cls_id", prev.get("cls_id", -1))),
                "cls_name": str(supp.get("cls_name", prev.get("cls_name", "unknown"))),
                "conf": float(supp.get("conf", prev.get("conf", 1.0))),
            }

        meta = track_meta.get(track_id, {"cls_id": -1, "cls_name": "unknown", "conf": 1.0})
        cls_name = meta["cls_name"]
        det_conf = meta["conf"]

        color = CLASS_COLORS.get(cls_name, DEFAULT_COLOR)

        x1i, y1i, x2i, y2i = map(int, [l, t, rgt, btm])
        cv2.rectangle(frame_vis, (x1i, y1i), (x2i, y2i), color, 3)

        # Show ONLY class name (no ID)
        draw_label(frame_vis, cls_name, x1i, y1i, color)
    font_scale = 0.4
    thickness = 1
    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    x1 = int(x)
    y2 = int(max(0, y))
    y1 = int(max(0, y2 - th - baseline - 4))
    x2 = int(x1 + tw + 6)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)
    cv2.putText(img, text, (x1 + 3, y2 - 3),
                font, font_scale, (0, 0, 0), thickness, cv2.LINE_AA)
        if WRITE_MOT:
            bb_left = float(l)
            bb_top = float(t)
            bb_width = float(bw)
            bb_height = float(bh)
            if MOT_1_BASED:
                bb_left += 1.0
                bb_top += 1.0

            mot_lines.append(
                f"{frame_idx},{track_id},{bb_left:.2f},{bb_top:.2f},{bb_width:.2f},{bb_height:.2f},{det_conf:.6f},-1,-1,-1"
            )

    if frame_idx % 100 == 0:
        print(
            f"Frame {frame_idx} | dets(after conf+NMS): {len(keep)} | tracks: {len(tracks)} | confirmed: {confirmed_count}"
        )

    return frame_vis


# Process first frame then remaining frames
writer.write(process_one_frame(first_frame))

while True:
    ok, frame = cap.read()
    if not ok:
        break

    # Ensure consistent size
    if frame.shape[1] != W or frame.shape[0] != H:
        frame = cv2.resize(frame, (W, H), interpolation=cv2.INTER_LINEAR)

    writer.write(process_one_frame(frame))

cap.release()
writer.release()

if WRITE_MOT:
    MOT_OUT.write_text("\n".join(mot_lines) + ("\n" if mot_lines else ""), encoding="utf-8")
    print(f"Saved MOT to: {MOT_OUT} (lines: {len(mot_lines)})")

print(f"OpenCV wrote temp video: {VIDEO_OUT_TMP}")

# Re-encode for Kaggle/Jupyter playback
subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(VIDEO_OUT_TMP),
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-crf", "18",
        "-preset", "fast",
        str(VIDEO_OUT_FINAL),
    ],
    check=True
)

print(f"Saved playable video: {VIDEO_OUT_FINAL}")


NameError: name 'text' is not defined

# Use this script after running the first three cells to get the tracking videos

In [7]:
from pathlib import Path
import subprocess
import cv2

from ultralytics import RTDETR
from deep_sort_realtime.deepsort_tracker import DeepSort


# =========================
# PATHS / CONFIG
# =========================
VIDEO_IN        = Path("/kaggle/input/datasets/huzaifasohail321/urban-intersection-4-15fps/UrbanIntersection4_15fps.mp4")
WEIGHTS         = Path("/kaggle/working/best.pt")
VIDEO_OUT_TMP   = Path("/kaggle/working/tracked_UrbanIntersection4_new_15fps.mp4")
VIDEO_OUT_FINAL = Path("/kaggle/working/tracked_UrbanIntersection4_new_15fps_final_h264.mp4")

# Per-class confidence thresholds
PER_CLASS_CONF = {
    "bus":         0.60,
    "car":         0.80,
    "truck":       0.60,
    "two_wheeler": 0.50,
    "van":         0.60,
}
DEFAULT_CONF          = 0.25
IMGSZ                 = 640
DRAW_DETECTIONS       = False
DRAW_UNCONFIRMED_TRACKS = False
NMS_IOU_THRES         = 0.50
MIN_BOX_PX            = 4
WRITE_MOT             = False
MOT_OUT               = Path("/kaggle/working/results.txt")
MOT_1_BASED           = True


# =========================
# COLORS (OpenCV BGR)
# =========================
CLASS_COLORS = {
    "bus":         (255,   0,   0),   # blue
    "car":         (  0, 255,   0),   # green
    "truck":       (  0,   0, 255),   # red
    "two_wheeler": (  0, 255, 255),   # yellow
    "van":         (255,   0, 255),   # magenta
}
DEFAULT_COLOR = (200, 200, 200)


# =========================
# HELPERS
# =========================
def draw_label(img, text, x, y, color):
    """Draw a filled-background label above the bounding box."""
    font       = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.4
    thickness  = 1
    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    x1 = int(x)
    y2 = int(max(0, y))
    y1 = int(max(0, y2 - th - baseline - 4))
    x2 = int(x1 + tw + 6)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)
    cv2.putText(img, text, (x1 + 3, y2 - 3),
                font, font_scale, (0, 0, 0), thickness, cv2.LINE_AA)


def nms_per_class(boxes_xywh, scores, cls_ids, score_thres, iou_thres):
    """Per-class NMS using cv2.dnn.NMSBoxes."""
    keep = []
    if not boxes_xywh:
        return keep
    for cid in sorted(set(cls_ids)):
        idxs  = [i for i, c in enumerate(cls_ids) if c == cid]
        b     = [[int(boxes_xywh[i][0]), int(boxes_xywh[i][1]),
                  int(boxes_xywh[i][2]), int(boxes_xywh[i][3])] for i in idxs]
        s     = [scores[i] for i in idxs]
        picked = cv2.dnn.NMSBoxes(b, s, score_threshold=score_thres, nms_threshold=iou_thres)
        if picked is None or len(picked) == 0:
            continue
        for p in picked:
            j = int(p[0]) if isinstance(p, (list, tuple)) else int(p)
            keep.append(idxs[j])
    keep.sort()
    return keep


# =========================
# LOAD MODEL + TRACKER
# =========================
detector = RTDETR(str(WEIGHTS))
tracker  = DeepSort(max_age=30, n_init=2)

cap = cv2.VideoCapture(str(VIDEO_IN))
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_IN}")

ok, first_frame = cap.read()
if not ok or first_frame is None:
    cap.release()
    raise RuntimeError("Could not read first frame from the input video.")

H, W = first_frame.shape[:2]
fps  = 15

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(VIDEO_OUT_TMP), fourcc, fps, (W, H))
if not writer.isOpened():
    cap.release()
    raise RuntimeError(f"Could not open VideoWriter for: {VIDEO_OUT_TMP}")

track_meta = {}
mot_lines  = []
frame_idx  = 0


# =========================
# MAIN PROCESSING FUNCTION
# =========================
def process_one_frame(frame):
    """Run detector + tracker on one frame; return annotated frame (BGR)."""
    global frame_idx, track_meta, mot_lines

    frame_idx += 1
    frame_vis  = frame.copy()

    # ── Detection ──────────────────────────────────────────────────────────
    results = detector(frame, imgsz=IMGSZ, verbose=False)
    r       = results[0]

    boxes_xywh, scores, cls_ids, cls_names = [], [], [], []

    if r.boxes is not None and len(r.boxes) > 0:
        for xyxy, conf, cls in zip(r.boxes.xyxy, r.boxes.conf, r.boxes.cls):
            conf   = float(conf)
            cls_id = int(cls)

            # resolve cls_name FIRST before using it in the confidence check
            cls_name = (r.names.get(cls_id, str(cls_id))
                        if isinstance(r.names, dict)
                        else (r.names[cls_id] if cls_id < len(r.names) else str(cls_id)))

            # per-class confidence filter
            if conf < PER_CLASS_CONF.get(cls_name, DEFAULT_CONF):
                continue

            x1, y1, x2, y2 = map(float, xyxy)
            w = x2 - x1
            h = y2 - y1

            if w <= 0 or h <= 0 or w < MIN_BOX_PX or h < MIN_BOX_PX:
                continue

            boxes_xywh.append([x1, y1, w, h])
            scores.append(conf)
            cls_ids.append(cls_id)
            cls_names.append(cls_name)

    # ── Per-class NMS ───────────────────────────────────────────────────────
    keep = nms_per_class(
        boxes_xywh, scores, cls_ids,
        min(PER_CLASS_CONF.values()),
        NMS_IOU_THRES
    )

    # ── Build DeepSORT inputs ───────────────────────────────────────────────
    dets, others = [], []
    for i in keep:
        x1, y1, w, h = boxes_xywh[i]
        conf         = scores[i]
        cls_id       = cls_ids[i]
        cls_name     = cls_names[i]

        dets.append(([x1, y1, w, h], conf, cls_id))
        others.append({"cls_id": cls_id, "cls_name": cls_name, "conf": conf})

        if DRAW_DETECTIONS:
            color = CLASS_COLORS.get(cls_name, DEFAULT_COLOR)
            cv2.rectangle(frame_vis,
                          (int(x1), int(y1)),
                          (int(x1 + w), int(y1 + h)),
                          color, 1)
            draw_label(frame_vis, f"{cls_name} {conf:.2f}", int(x1), int(y1), color)

    # ── Tracker update ──────────────────────────────────────────────────────
    try:
        tracks = tracker.update_tracks(dets, frame=frame, others=others)
    except TypeError:
        tracks = tracker.update_tracks(dets, frame=frame)

    confirmed_count = 0

    # ── Draw confirmed tracks ───────────────────────────────────────────────
    for trk in tracks:
        if trk.is_confirmed():
            confirmed_count += 1

        if not DRAW_UNCONFIRMED_TRACKS and not trk.is_confirmed():
            continue

        if hasattr(trk, "time_since_update") and trk.time_since_update > 0:
            continue

        track_id        = trk.track_id
        l, t, rgt, btm = trk.to_ltrb()

        l   = max(0.0, min(float(l),   W - 1.0))
        t   = max(0.0, min(float(t),   H - 1.0))
        rgt = max(0.0, min(float(rgt), W - 1.0))
        btm = max(0.0, min(float(btm), H - 1.0))

        bw = rgt - l
        bh = btm - t
        if bw < MIN_BOX_PX or bh < MIN_BOX_PX:
            continue

        # Update track metadata cache
        supp = trk.get_det_supplementary() if hasattr(trk, "get_det_supplementary") else None
        if isinstance(supp, dict) and ("cls_id" in supp or "cls_name" in supp):
            prev = track_meta.get(track_id, {})
            track_meta[track_id] = {
                "cls_id":   int(supp.get("cls_id",   prev.get("cls_id",   -1))),
                "cls_name": str(supp.get("cls_name", prev.get("cls_name", "unknown"))),
                "conf":   float(supp.get("conf",     prev.get("conf",     1.0))),
            }

        meta     = track_meta.get(track_id, {"cls_id": -1, "cls_name": "unknown", "conf": 1.0})
        cls_name = meta["cls_name"]
        det_conf = meta["conf"]
        color    = CLASS_COLORS.get(cls_name, DEFAULT_COLOR)

        x1i, y1i, x2i, y2i = map(int, [l, t, rgt, btm])

        cv2.rectangle(frame_vis, (x1i, y1i), (x2i, y2i), color, 1)
        draw_label(frame_vis, f"ID:{track_id} {cls_name} {det_conf:.2f}", x1i, y1i, color)

        if WRITE_MOT:
            bb_left = float(l) + (1.0 if MOT_1_BASED else 0.0)
            bb_top  = float(t) + (1.0 if MOT_1_BASED else 0.0)
            mot_lines.append(
                f"{frame_idx},{track_id},{bb_left:.2f},{bb_top:.2f},"
                f"{bw:.2f},{bh:.2f},{det_conf:.6f},-1,-1,-1"
            )

    if frame_idx % 100 == 0:
        print(f"Frame {frame_idx} | dets(after NMS): {len(keep)} "
              f"| tracks: {len(tracks)} | confirmed: {confirmed_count}")

    return frame_vis


# =========================
# RUN
# =========================
writer.write(process_one_frame(first_frame))

while True:
    ok, frame = cap.read()
    if not ok:
        break
    if frame.shape[1] != W or frame.shape[0] != H:
        frame = cv2.resize(frame, (W, H), interpolation=cv2.INTER_LINEAR)
    writer.write(process_one_frame(frame))

cap.release()
writer.release()

if WRITE_MOT:
    MOT_OUT.write_text("\n".join(mot_lines) + ("\n" if mot_lines else ""), encoding="utf-8")
    print(f"Saved MOT to: {MOT_OUT} (lines: {len(mot_lines)})")

print(f"OpenCV wrote temp video: {VIDEO_OUT_TMP}")

# =========================
# RE-ENCODE WITH FFMPEG
# =========================
subprocess.run(
    [
        "ffmpeg", "-y",
        "-i",        str(VIDEO_OUT_TMP),
        "-c:v",      "libx264",
        "-pix_fmt",  "yuv420p",
        "-movflags", "+faststart",
        "-crf",      "18",
        "-preset",   "fast",
        str(VIDEO_OUT_FINAL),
    ],
    check=True
)

print(f"Saved playable video: {VIDEO_OUT_FINAL}")

Frame 100 | dets(after NMS): 5 | tracks: 5 | confirmed: 5
Frame 200 | dets(after NMS): 7 | tracks: 8 | confirmed: 8
Frame 300 | dets(after NMS): 4 | tracks: 5 | confirmed: 5
Frame 400 | dets(after NMS): 6 | tracks: 6 | confirmed: 6
OpenCV wrote temp video: /kaggle/working/tracked_UrbanIntersection4_new_15fps.mp4


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Saved playable video: /kaggle/working/tracked_UrbanIntersection4_new_15fps_final_h264.mp4


[mp4 @ 0x597d04d01680] Starting second pass: moving the moov atom to the beginning of the file
frame=  476 fps= 32 q=-1.0 Lsize=   20557kB time=00:00:31.53 bitrate=5340.4kbits/s speed=2.15x    
video:20551kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.028897%
[libx264 @ 0x597d04cff7c0] frame I:2     Avg QP:14.69  size:190510
[libx264 @ 0x597d04cff7c0] frame P:226   Avg QP:16.30  size: 73015
[libx264 @ 0x597d04cff7c0] frame B:248   Avg QP:21.35  size: 16778
[libx264 @ 0x597d04cff7c0] consecutive B-frames: 28.8%  3.8%  4.4% 63.0%
[libx264 @ 0x597d04cff7c0] mb I  I16..4:  8.4% 78.6% 13.0%
[libx264 @ 0x597d04cff7c0] mb P  I16..4:  0.7%  2.9%  1.1%  P16..4: 43.5% 23.6% 19.1%  0.0%  0.0%    skip: 9.1%
[libx264 @ 0x597d04cff7c0] mb B  I16..4:  0.3%  1.4%  0.1%  B16..8: 44.1% 16.6%  2.1%  direct:11.4%  skip:23.9%  L0:36.3% L1:30.3% BI:33.3%
[libx264 @ 0x597d04cff7c0] 8x8 transform intra:67.0% inter:73.8%
[libx264 @ 0x597d04cff7c0] coded y,uvDC,uvAC intra: 65.

In [11]:
from IPython.display import Video
Video("/kaggle/working/tracked_RoundaboutNear1_15fps_final.mp4", embed=True)

# use this script to get the annotated tracking frames to display the tracking data files

In [4]:
import zipfile
import re
import cv2
import numpy as np
from pathlib import Path

from ultralytics import RTDETR
from deep_sort_realtime.deepsort_tracker import DeepSort


# =========================
# PATHS / CONFIG
# =========================
FRAMES_IN_DIR  = Path("/kaggle/input/datasets/huzaifasohail321/regional-road-frames/RegionalRoad_Frames")
ZIP_OUT        = Path("/kaggle/working/annotated_frames_regional_road.zip")
WEIGHTS        = Path("/kaggle/working/best.pt")

# Per-class confidence thresholds
PER_CLASS_CONF = {
    "bus":         0.60,
    "car":         0.80,
    "truck":       0.60,
    "two_wheeler": 0.50,
    "van":         0.60,
}
DEFAULT_CONF            = 0.25
IMGSZ                   = 640
DRAW_DETECTIONS         = False
DRAW_UNCONFIRMED_TRACKS = False
NMS_IOU_THRES           = 0.50
MIN_BOX_PX              = 4


# =========================
# COLORS (OpenCV BGR)
# =========================
CLASS_COLORS = {
    "bus":         (255,   0,   0),
    "car":         (  0, 255,   0),
    "truck":       (  0,   0, 255),
    "two_wheeler": (  0, 255, 255),
    "van":         (255,   0, 255),
}
DEFAULT_COLOR = (200, 200, 200)


# =========================
# HELPERS
# =========================
def draw_label(img, text, x, y, color):
    font       = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.4
    thickness  = 1
    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    x1 = int(x)
    y2 = int(max(0, y))
    y1 = int(max(0, y2 - th - baseline - 4))
    x2 = int(x1 + tw + 6)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)
    cv2.putText(img, text, (x1 + 3, y2 - 3),
                font, font_scale, (0, 0, 0), thickness, cv2.LINE_AA)


def nms_per_class(boxes_xywh, scores, cls_ids, score_thres, iou_thres):
    keep = []
    if not boxes_xywh:
        return keep
    for cid in sorted(set(cls_ids)):
        idxs   = [i for i, c in enumerate(cls_ids) if c == cid]
        b      = [[int(boxes_xywh[i][0]), int(boxes_xywh[i][1]),
                   int(boxes_xywh[i][2]), int(boxes_xywh[i][3])] for i in idxs]
        s      = [scores[i] for i in idxs]
        picked = cv2.dnn.NMSBoxes(b, s, score_threshold=score_thres, nms_threshold=iou_thres)
        if picked is None or len(picked) == 0:
            continue
        for p in picked:
            j = int(p[0]) if isinstance(p, (list, tuple)) else int(p)
            keep.append(idxs[j])
    keep.sort()
    return keep


def sort_key(p: Path):
    numbers = re.findall(r'\d+', p.stem)
    return int(numbers[-1]) if numbers else p.stem


# =========================
# LOAD MODEL + TRACKER
# =========================
detector = RTDETR(str(WEIGHTS))
tracker  = DeepSort(max_age=30, n_init=2)

# =========================
# COLLECT + SORT FRAMES
# =========================
extensions  = ('.jpg', '.jpeg', '.png', '.bmp')
frame_paths = [f for f in FRAMES_IN_DIR.iterdir()
               if f.suffix.lower() in extensions]
frame_paths = sorted(frame_paths, key=sort_key)

if not frame_paths:
    raise FileNotFoundError(f"No image files found in {FRAMES_IN_DIR}")

print(f"Found {len(frame_paths)} frames in {FRAMES_IN_DIR}")
print(f"Writing annotated frames directly to zip: {ZIP_OUT}\n")

# Delete existing zip if present
if ZIP_OUT.exists():
    ZIP_OUT.unlink()
    print(f"Deleted existing zip: {ZIP_OUT}")

# =========================
# STATE
# =========================
track_meta = {}
frame_idx  = 0

# =========================
# PROCESS + ZIP DIRECTLY
# =========================
with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as zf:

    for img_path in frame_paths:
        frame = cv2.imread(str(img_path))
        if frame is None:
            print(f"  [WARN] Could not read {img_path.name}, skipping.")
            continue

        frame_idx += 1
        H, W      = frame.shape[:2]
        frame_vis = frame.copy()

        # ── Detection ────────────────────────────────────────────────────
        results = detector(frame, imgsz=IMGSZ, verbose=False)
        r       = results[0]

        boxes_xywh, scores, cls_ids, cls_names = [], [], [], []

        if r.boxes is not None and len(r.boxes) > 0:
            for xyxy, conf, cls in zip(r.boxes.xyxy, r.boxes.conf, r.boxes.cls):
                conf   = float(conf)
                cls_id = int(cls)

                cls_name = (r.names.get(cls_id, str(cls_id))
                            if isinstance(r.names, dict)
                            else (r.names[cls_id] if cls_id < len(r.names) else str(cls_id)))

                if conf < PER_CLASS_CONF.get(cls_name, DEFAULT_CONF):
                    continue

                x1, y1, x2, y2 = map(float, xyxy)
                w = x2 - x1
                h = y2 - y1

                if w <= 0 or h <= 0 or w < MIN_BOX_PX or h < MIN_BOX_PX:
                    continue

                boxes_xywh.append([x1, y1, w, h])
                scores.append(conf)
                cls_ids.append(cls_id)
                cls_names.append(cls_name)

        # ── Per-class NMS ────────────────────────────────────────────────
        keep = nms_per_class(
            boxes_xywh, scores, cls_ids,
            min(PER_CLASS_CONF.values()),
            NMS_IOU_THRES
        )

        # ── Build DeepSORT inputs ────────────────────────────────────────
        dets, others = [], []
        for i in keep:
            x1, y1, w, h = boxes_xywh[i]
            conf         = scores[i]
            cls_id       = cls_ids[i]
            cls_name     = cls_names[i]

            dets.append(([x1, y1, w, h], conf, cls_id))
            others.append({"cls_id": cls_id, "cls_name": cls_name, "conf": conf})

            if DRAW_DETECTIONS:
                color = CLASS_COLORS.get(cls_name, DEFAULT_COLOR)
                cv2.rectangle(frame_vis,
                              (int(x1), int(y1)),
                              (int(x1 + w), int(y1 + h)),
                              color, 1)
                draw_label(frame_vis, f"{cls_name} {conf:.2f}",
                           int(x1), int(y1), color)

        # ── Tracker update ───────────────────────────────────────────────
        try:
            tracks = tracker.update_tracks(dets, frame=frame, others=others)
        except TypeError:
            tracks = tracker.update_tracks(dets, frame=frame)

        # ── Draw confirmed tracks ────────────────────────────────────────
        for trk in tracks:
            if not DRAW_UNCONFIRMED_TRACKS and not trk.is_confirmed():
                continue

            if hasattr(trk, "time_since_update") and trk.time_since_update > 0:
                continue

            track_id        = trk.track_id
            l, t, rgt, btm = trk.to_ltrb()

            l   = max(0.0, min(float(l),   W - 1.0))
            t   = max(0.0, min(float(t),   H - 1.0))
            rgt = max(0.0, min(float(rgt), W - 1.0))
            btm = max(0.0, min(float(btm), H - 1.0))

            bw = rgt - l
            bh = btm - t
            if bw < MIN_BOX_PX or bh < MIN_BOX_PX:
                continue

            supp = trk.get_det_supplementary() if hasattr(trk, "get_det_supplementary") else None
            if isinstance(supp, dict) and ("cls_id" in supp or "cls_name" in supp):
                prev = track_meta.get(track_id, {})
                track_meta[track_id] = {
                    "cls_id":   int(supp.get("cls_id",   prev.get("cls_id",   -1))),
                    "cls_name": str(supp.get("cls_name", prev.get("cls_name", "unknown"))),
                    "conf":   float(supp.get("conf",     prev.get("conf",     1.0))),
                }

            meta     = track_meta.get(track_id,
                                      {"cls_id": -1, "cls_name": "unknown", "conf": 1.0})
            cls_name = meta["cls_name"]
            det_conf = meta["conf"]
            color    = CLASS_COLORS.get(cls_name, DEFAULT_COLOR)

            x1i, y1i, x2i, y2i = map(int, [l, t, rgt, btm])
            cv2.rectangle(frame_vis, (x1i, y1i), (x2i, y2i), color, 1)
            draw_label(frame_vis,
                       f"ID:{track_id} {cls_name} {det_conf:.2f}",
                       x1i, y1i, color)

        # ── Encode frame to PNG bytes and write directly into zip ────────
        success, buffer = cv2.imencode('.png', frame_vis)
        if success:
            zf.writestr(img_path.stem + ".png", buffer.tobytes())

        if frame_idx % 50 == 0:
            print(f"  Processed {frame_idx}/{len(frame_paths)} frames...")

print(f"\nDone. {frame_idx} annotated frames zipped to {ZIP_OUT}")
print(f"Size: {ZIP_OUT.stat().st_size / (1024 * 1024):.1f} MB")
print("Download from the Kaggle Output panel on the right.")

Found 4500 frames in /kaggle/input/datasets/huzaifasohail321/regional-road-frames/RegionalRoad_Frames
Writing annotated frames directly to zip: /kaggle/working/annotated_frames_regional_road.zip

  Processed 50/4500 frames...
  Processed 100/4500 frames...
  Processed 150/4500 frames...
  Processed 200/4500 frames...
  Processed 250/4500 frames...
  Processed 300/4500 frames...
  Processed 350/4500 frames...
  Processed 400/4500 frames...
  Processed 450/4500 frames...
  Processed 500/4500 frames...
  Processed 550/4500 frames...
  Processed 600/4500 frames...
  Processed 650/4500 frames...
  Processed 700/4500 frames...
  Processed 750/4500 frames...
  Processed 800/4500 frames...
  Processed 850/4500 frames...
  Processed 900/4500 frames...
  Processed 950/4500 frames...
  Processed 1000/4500 frames...
  Processed 1050/4500 frames...
  Processed 1100/4500 frames...
  Processed 1150/4500 frames...
  Processed 1200/4500 frames...
  Processed 1250/4500 frames...
  Processed 1300/4500 fr

In [9]:
import subprocess
subprocess.run([
    "zip", "-r",
    "/kaggle/working/annotated_frames_regional_road.zip",
    "/kaggle/working/annotated_frames_regional_road"
], check=True)
print("Zipped successfully.")

  adding: kaggle/working/annotated_frames_regional_road/ (stored 0%)
  adding: kaggle/working/annotated_frames_regional_road/frame_003102.png (deflated 7%)
  adding: kaggle/working/annotated_frames_regional_road/frame_000411.png (deflated 6%)
  adding: kaggle/working/annotated_frames_regional_road/frame_002781.png (deflated 8%)
  adding: kaggle/working/annotated_frames_regional_road/frame_002217.png (deflated 7%)
  adding: kaggle/working/annotated_frames_regional_road/frame_004402.png (deflated 7%)
  adding: kaggle/working/annotated_frames_regional_road/frame_001189.png (deflated 6%)
  adding: kaggle/working/annotated_frames_regional_road/frame_003290.png (deflated 7%)
  adding: kaggle/working/annotated_frames_regional_road/frame_004323.png (deflated 7%)
  adding: kaggle/working/annotated_frames_regional_road/frame_002395.png (deflated 6%)
  adding: kaggle/working/annotated_frames_regional_road/frame_000337.png (deflated 6%)
  adding: kaggle/working/annotated_frames_regional_road/frame

CalledProcessError: Command '['zip', '-r', '/kaggle/working/annotated_frames_regional_road.zip', '/kaggle/working/annotated_frames_regional_road']' returned non-zero exit status 14.

# To check whether Detector is working or not

In [ ]:
# import os
# import cv2

# out_path = "/kaggle/working/tracked_RoundaboutNear3_new.mp4"

# print("Exists:", os.path.exists(out_path))
# if os.path.exists(out_path):
#     print("Size (MB):", os.path.getsize(out_path) / (1024 * 1024))

# cap = cv2.VideoCapture(out_path)
# print("OpenCV can open:", cap.isOpened())

# ok, fr = cap.read()
# print("Read first frame:", ok, "shape:", None if fr is None else fr.shape)

# cap.release()

In [ ]:
# import cv2
# from ultralytics import RTDETR

# VIDEO_IN = "/kaggle/input/datasets/huzaifasohail321/roundaboutnear3-video/RoundaboutNear3.mp4"
# WEIGHTS  = "/kaggle/working/best.pt"

# detector = RTDETR(WEIGHTS)

# cap = cv2.VideoCapture(VIDEO_IN)
# ok, frame = cap.read()
# cap.release()

# print("Read frame:", ok, "shape:", None if frame is None else frame.shape)

# res = detector(frame)[0]
# n = 0 if res.boxes is None else len(res.boxes)
# print("Detections on first frame:", n)

# if n:
#     print("Top conf:", float(res.boxes.conf.max()))
#     # show a few class ids
#     print("Some cls ids:", [int(x) for x in res.boxes.cls[:10].tolist()])
#     print("names:", res.names)


In [ ]:
from pathlib import Path
import re
import cv2
import subprocess

# =========================
# CONFIG (EDIT THESE)
# =========================
FRAMES_DIR = Path("/kaggle/input/datasets/huzaifasohail321/splitted-roundabout-frames/Splitted_roundabout_Frames")  # folder containing images
OUTPUT_TMP = Path("/kaggle/working/splitted-roundabout_tmp.mp4")       # OpenCV output
OUTPUT_MP4 = Path("/kaggle/working/splitted-roundabout_h264.mp4")      # final playable output

FPS = 15  # <-- IMPORTANT: set to the real fps you want (e.g., 15, 25, 30)

# Allowed image extensions
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

# Optional: if your files are like frame_00001.jpg, this extracts 1,2,3...
NUM_RE = re.compile(r"(\d+)")


def get_frame_number(p: Path) -> int:
    """
    Extract first number from filename stem for numeric sorting.
    Example: frame_00012.jpg -> 12
    If no number exists, fallback to a large number so it sorts last.
    """
    m = NUM_RE.search(p.stem)
    return int(m.group(1)) if m else 10**18


# =========================
# COLLECT + SORT FRAMES
# =========================
frame_paths = [p for p in FRAMES_DIR.iterdir() if p.suffix.lower() in IMG_EXTS]
frame_paths.sort(key=lambda p: (get_frame_number(p), p.name))

if not frame_paths:
    raise FileNotFoundError(f"No frames found in: {FRAMES_DIR}")

print("Frames found:", len(frame_paths))
print("First frame:", frame_paths[0].name)
print("Last frame:", frame_paths[-1].name)

# =========================
# INIT VIDEO WRITER
# =========================
first = cv2.imread(str(frame_paths[0]))
if first is None:
    raise RuntimeError(f"Could not read first frame: {frame_paths[0]}")

H, W = first.shape[:2]
print("Frame size:", (W, H))
print("FPS:", FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")  # temp codec
writer = cv2.VideoWriter(str(OUTPUT_TMP), fourcc, float(FPS), (W, H))
if not writer.isOpened():
    raise RuntimeError("Could not open VideoWriter. Try a different codec or check OpenCV build.")

# =========================
# WRITE VIDEO
# =========================
bad = 0
for i, p in enumerate(frame_paths, start=1):
    img = cv2.imread(str(p))
    if img is None:
        bad += 1
        continue

    # Ensure consistent size
    if img.shape[1] != W or img.shape[0] != H:
        img = cv2.resize(img, (W, H), interpolation=cv2.INTER_LINEAR)

    writer.write(img)

    if i % 500 == 0:
        print(f"Wrote {i}/{len(frame_paths)} frames...")

writer.release()
print("OpenCV temp video written:", OUTPUT_TMP)
if bad:
    print("WARNING: unreadable frames skipped:", bad)

# =========================
# RE-ENCODE FOR KAGGLE PLAYBACK (RECOMMENDED)
# =========================
# This typically fixes "black video / not playing" issues in notebook players.
try:
    subprocess.run(
        [
            "ffmpeg", "-y",
            "-i", str(OUTPUT_TMP),
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            "-crf", "18",
            "-preset", "fast",
            str(OUTPUT_MP4),
        ],
        check=True
    )
    print("Final playable video written:", OUTPUT_MP4)
except Exception as e:
    print("ffmpeg re-encode failed (you can still try playing the temp mp4):", e)
    print("Temp video:", OUTPUT_TMP)


In [ ]:
from IPython.display import Video
Video("/kaggle/working/splitted-roundabout_h264.mp4", embed=True)